## Step 1 : install and load the libs

In [1]:
#install the libs

!pip install -q transformers peft bitsandbytes trl datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 44.9 MB/s eta 0:00:00


In [2]:
!pip install bitsandbytes==0.46.1 --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 23.4 MB/s eta 0:00:00:00:0100:01


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [4]:
#import the required libs

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

In [5]:
# adding the hf token and login

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
login(token=secret.get_secret("HF_TOKEN_LLAMA"))
if login : 
    print("login valid")
else : 
    print("login failed")

login valid


## step 2 : Load the data from hf and write the prompt format for training

In [6]:
# loading the dataset from hf

dataset = load_dataset("spider")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})


In [7]:
print(dataset["train"][0])

{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'question': 'How many heads of the departments are older than 56 ?', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?']}


In [8]:
# engineered prompt for training and inference

def format_prompt(example):
    return {
        "text": f"""### Task
Generate a SQL query based on the question and schema below.

### Schema
{example['db_id']}

### Question
{example['question']}

### SQL
{example['query']}"""
    }

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

In [9]:
print(dataset.keys())

dict_keys(['train', 'validation'])


In [10]:
print(dataset["train"][0])

{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'question': 'How many heads of the departments are older than 56 ?', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?'], 'text': '### Task\nGenerate a SQL query based on the question and schema below.\n\n### Schema\ndepartment_management\n\n### Question\nHow many heads of the departments are older than 56 ?\n\n### SQL\nSELECT count(*) FROM head WHERE age  >  56'}


In [11]:
for i in range(3):
    print(f"--- Sample {i+1} ---")
    print("Question:", dataset['train'][i]['question'])
    print("SQL:", dataset['train'][i]['query'])
    print("DB:", dataset['train'][i]['db_id'])
    print()

--- Sample 1 ---
Question: How many heads of the departments are older than 56 ?
SQL: SELECT count(*) FROM head WHERE age  >  56
DB: department_management

--- Sample 2 ---
Question: List the name, born state and age of the heads of departments ordered by age.
SQL: SELECT name ,  born_state ,  age FROM head ORDER BY age
DB: department_management

--- Sample 3 ---
Question: List the creation year, name and budget of each department.
SQL: SELECT creation ,  name ,  budget_in_billions FROM department
DB: department_management



In [12]:
dataset = dataset.map(format_prompt)
print(dataset['train'][0]['text'])

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

### Task
Generate a SQL query based on the question and schema below.

### Schema
department_management

### Question
How many heads of the departments are older than 56 ?

### SQL
SELECT count(*) FROM head WHERE age  >  56


## step 3 : Load the model and tokenizer from the  and quantized it


In [13]:
# configure the quantization

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [14]:
# adding this to clear memory as loading a 16gb model in the follwoing cell

import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [15]:
model_name = "meta-llama/Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True

)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

## step 4 : Lora Adapter Configuration(PEFT)

In [16]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

#prepare model for kbit trainig, 1)freeze base layers 2)gradeint checkpoint
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r = 16,               #4096x16 16x4096
    lora_alpha = 32,      # standard twice as r
    target_modules = [    # attention layers are targeted only since most of the relations building is learned here i.e between language and sql
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout = 0.05,   #regularization techniqe used for preventing overfitting
    bias = "none",         # just not worth training
    task_type = "CAUSAL_LM"
)

model = get_peft_model(model, lora_config)   # just attaching the layers A and B to the oringinal linear layer but it has no effect intially as its intialized with 0

model.print_trainable_parameters() 

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


## Step 5a: SFTTrainer Setup
its a trl library built on hugging face, controls the entire training loop, forward pass,loss computation , gradient accumalation and checkpointing

In [22]:
#test code 
import trl
print(trl.__version__)

1.2.0


In [28]:
from trl import SFTTrainer, SFTConfig

# trainig config
training_args = SFTConfig(
    output_dir = "./qlora-text2sql",
    num_train_epochs = 3,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10, #print loss after every 10 steps
    eval_strategy="steps", # one step is after how many samples the weight get updates so batch_size x gradient_accumalation steps
    eval_steps=100,
    save_strategy="steps",
    load_best_model_at_end=True,
    dataset_text_field="text",
    report_to="none",
    packing=False
)

#intialize the trainer
trainer = SFTTrainer(
    model = model,
    args = training_args,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    processing_class = tokenizer

)

## Step 5b : MLflow integration

In [26]:
#install mlflow
!pip install mlflow --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 715.8 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 35.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.5/857.5 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.7 MB/s eta 0:00:00


In [ ]:
import mlflow
from transformers import TrainerCallback

# make a callback function so it can bridge the trainer logs with mlflow
class MLflowCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        # `logs` is a dict like {"loss": 0.43, "eval_loss": 0.38, "epoch": 1.2}
        # `state.global_step` is the current step number
        if logs:
            for key, value in logs.items():
                if isinstance(value, (int, float)):  # skip non-numeric
                    mlflow.log_metric(key, value, step=state.global_step)

# --- MLflow setup ---
mlflow.set_tracking_uri("file:///kaggle/working/mlruns")  # save locally
mlflow.set_experiment("qlora-text2sql")

# --- Wrap training in a run ---
with mlflow.start_run(run_name="llama3.1-8b-spider-v1"):

    # Log all your hyperparams once
    mlflow.log_params({
        "model_name": "meta-llama/Meta-Llama-3.1-8B",
        "learning_rate": 2e-4,
        "batch_size": 4,
        "gradient_accumulation_steps": 4,
        "num_epochs": 3,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "max_seq_length": 512,
    })

    # Attach MLflow callback to trainer
    trainer.add_callback(MLflowCallback())

    # Train
    trainer.train()

    # Save adapter + tokenizer
    trainer.model.save_pretrained("./final_adapter")
    tokenizer.save_pretrained("./final_adapter")

    # Log the saved folder as artifact
    mlflow.log_artifacts("./final_adapter", artifact_path="final_adapter")

print("Training done. MLflow run saved to /kaggle/working/mlruns")

Step,Training Loss,Validation Loss
100,0.702048,0.920338
200,0.630031,0.945375
300,0.579118,0.981879
400,0.538170,1.033129
500,0.484495,1.039175
600,0.469672,1.080059
700,0.446338,1.109937
800,0.435259,1.100693


In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import shutil

secret = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN_LLAMA")
login(token=hf_token)

api = HfApi()

# push adapter
api.upload_folder(
    folder_path="./final_adapter",
    repo_id="Awan8754/llama3.1-text2sql-adapter",
    repo_type="model",
    token=hf_token
)

# zip and push mlruns
shutil.make_archive("mlruns_backup", "zip", "/kaggle/working/mlruns")
api.upload_file(
    path_or_fileobj="mlruns_backup.zip",
    path_in_repo="mlruns_backup.zip",
    repo_id="Awan8754/llama3.1-text2sql-adapter",
    repo_type="model",
    token=hf_token
)

print("Everything saved to HF. Sleep well.")